# Machine Learning Pipeline
Pipeline pelatihan model peramalan harga saham temporal menggunakan arsitektur **LSTM** dan **GRU**.
Normalisasi skala diekspor ke format JSON bahasa-agnostik dan dataset sekuensial dialirkan secara native via .

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.callbacks import EarlyStopping
from keras.layers import Dense, Dropout, GRU, LSTM
from keras.models import Sequential
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler


## 1. Konfigurasi Parameter Global


In [ ]:
TICKER = "BBCA.JK"
TARGET_COL = "close"
TRAIN_SIZE_RATIO = 0.80
RANDOM_STATE = 42

# Hiperparameter Deep Learning
WINDOW_SIZE = 30
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001


## 2. Data Understanding


### 2.1. Memuat Data


In [ ]:
# Path Penyimpanan Artefak
DATA_PATH = Path(f"artifact/data/{TICKER}.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"File data tidak ditemukan di: {DATA_PATH}")

OUTPUT_DIR = Path("artifact/model")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date").reset_index(drop=True)
df.head()


### 2.2. Mengecek Data Hilang


In [ ]:
print(df.isna().sum())


### 2.3. Pengecekan Gap Tanggal


In [ ]:
full_date_range = pd.date_range(start=df["date"].min(), end=df["date"].max(), freq="D")
missing_dates = full_date_range.difference(df["date"])
print(f"Jumlah Hari Libur Bursa / Non-Trading Days: {len(missing_dates)}")


### 2.4. Visualisasi Tren Harga Historis


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["date"], df[TARGET_COL], label=f"Actual {TARGET_COL.capitalize()}", color="royalblue")
plt.title(f"Tren Historis Saham - {TICKER}")
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


## 3. Data Preparation


### 3.2. Pembagian Data Temporal


In [ ]:
train_len = int(len(df) * TRAIN_SIZE_RATIO)
train_df = df.iloc[:train_len].copy().reset_index(drop=True)
test_df = df.iloc[train_len:].copy().reset_index(drop=True)

print(f"Jumlah Sampel Latih : {len(train_df)}")
print(f"Jumlah Sampel Uji   : {len(test_df)}")


### 3.3. Normalisasi Skala


In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_df[[TARGET_COL]].values)
test_scaled = scaler.transform(test_df[[TARGET_COL]].values)


### 3.4. Penyiapan Dataset Sekuensial Native (Timeseries Dataset)
Menggunakan  secara native tanpa loop manual Python, mengoptimalkan throughput data dan mempercepat pelatihan.

In [ ]:
# Dataset Latih: Windowing native TensorFlow (tanpa step sliding window manual python loop)
train_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=train_scaled[:-1],
    targets=train_scaled[WINDOW_SIZE:, 0],
    sequence_length=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=RANDOM_STATE
)

# Dataset Uji: Disambung dari titik akhir data latih agar seluruh baris test_df terprediksi
test_window_inputs = np.concatenate([train_scaled[-WINDOW_SIZE:], test_scaled], axis=0)
test_dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=test_window_inputs[:-1],
    targets=test_window_inputs[WINDOW_SIZE:, 0],
    sequence_length=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"Jumlah Batch Latih : {len(train_dataset)}")
print(f"Jumlah Batch Uji   : {len(test_dataset)}")


## 4. Modeling (LSTM & GRU)


In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


In [ ]:
early_stopping = EarlyStopping(
    monitor="loss",
    patience=8,
    restore_best_weights=True,
    verbose=0
)


### 4.1. Pelatihan Model LSTM (Long Short-Term Memory)


In [ ]:
lstm_model = Sequential([
    LSTM(units=64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2),
    LSTM(units=32, return_sequences=False),
    Dropout(0.2),
    Dense(units=16, activation="relu"),
    Dense(units=1)
], name="Stock_LSTM_Model")

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mean_squared_error"
)

print("Memulai pelatihan model LSTM...")
lstm_history = lstm_model.fit(
    train_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1
)


### 4.2. Pelatihan Model GRU (Gated Recurrent Unit)


In [ ]:
gru_model = Sequential([
    GRU(units=64, return_sequences=True, input_shape=(WINDOW_SIZE, 1)),
    Dropout(0.2),
    GRU(units=32, return_sequences=False),
    Dropout(0.2),
    Dense(units=16, activation="relu"),
    Dense(units=1)
], name="Stock_GRU_Model")

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="mean_squared_error"
)

print("Memulai pelatihan model GRU...")
gru_history = gru_model.fit(
    train_dataset,
    epochs=EPOCHS,
    callbacks=[early_stopping],
    verbose=1
)


## 5. Evaluation


### 5.1. Inferensi Data Uji


In [ ]:
lstm_preds_scaled = lstm_model.predict(test_dataset, verbose=0)
lstm_forecast_vals = scaler.inverse_transform(lstm_preds_scaled).flatten()

gru_preds_scaled = gru_model.predict(test_dataset, verbose=0)
gru_forecast_vals = scaler.inverse_transform(gru_preds_scaled).flatten()

y_true = test_df[TARGET_COL].values


### 5.2. Kalkulasi Metrik Evaluasi


In [ ]:
def calculate_metrics(actual, predicted):
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    return {"RMSE": float(rmse), "MAPE": float(mape), "R2": float(r2)}

metrics_lstm = calculate_metrics(y_true, lstm_forecast_vals)
metrics_gru = calculate_metrics(y_true, gru_forecast_vals)

print("--- Hasil Evaluasi LSTM ---")
for k, v in metrics_lstm.items():
    print(f"{k}: {v:.4f}")

print("\n--- Hasil Evaluasi GRU ---")
for k, v in metrics_gru.items():
    print(f"{k}: {v:.4f}")

best_variant = "LSTM" if metrics_lstm["RMSE"] <= metrics_gru["RMSE"] else "GRU"
print(f"\nVarian Model Terbaik: {best_variant}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

# Subplot 1: LSTM
axes[0].plot(train_df["date"], train_df[TARGET_COL], label="Data Latih (Aktual)", alpha=0.5)
axes[0].plot(test_df["date"], y_true, label="Data Uji (Aktual)", alpha=0.5)
axes[0].plot(test_df["date"], lstm_forecast_vals, label="Prediksi LSTM", color="green")
axes[0].set_title(f"Evaluasi LSTM - {TICKER}")
axes[0].set_ylabel("Harga (Rp)")
axes[0].legend(loc="upper left")
axes[0].grid(True, linestyle="--", alpha=0.5)

# Subplot 2: GRU
axes[1].plot(train_df["date"], train_df[TARGET_COL], label="Data Latih (Aktual)", alpha=0.5)
axes[1].plot(test_df["date"], y_true, label="Data Uji (Aktual)", alpha=0.5)
axes[1].plot(test_df["date"], gru_forecast_vals, label="Prediksi GRU", color="darkorange")
axes[1].set_title(f"Evaluasi GRU - {TICKER}")
axes[1].set_xlabel("Tanggal")
axes[1].set_ylabel("Harga (Rp)")
axes[1].legend(loc="upper left")
axes[1].grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()


## 6. Penyimpanan Model dan Artefak


In [ ]:
# 1. Simpan Model Deep Learning (Format Keras)
lstm_path = OUTPUT_DIR / f"{TICKER}_LSTM.keras"
gru_path = OUTPUT_DIR / f"{TICKER}_GRU.keras"

lstm_model.save(lstm_path)
gru_model.save(gru_path)
print(f"Model LSTM berhasil disimpan ke : {lstm_path}")
print(f"Model GRU berhasil disimpan ke  : {gru_path}")

# 2. Simpan Parameter Scaler ke JSON (Bahasa-Agnostik)
scaler_data = {
    "scaler_type": "MinMaxScaler",
    "ticker": TICKER,
    "target_col": TARGET_COL,
    "feature_range": list(scaler.feature_range),
    "data_min": float(scaler.data_min_[0]),
    "data_max": float(scaler.data_max_[0]),
    "data_range": float(scaler.data_range_[0]),
    "scale": float(scaler.scale_[0]),
    "min": float(scaler.min_[0]),
    "n_samples_seen": int(scaler.n_samples_seen_),
    "train_samples": len(train_df),
    "total_samples": len(df),
}
scaler_path = OUTPUT_DIR / f"{TICKER}_scaler.json"
with open(scaler_path, "w", encoding="utf-8") as f:
    json.dump(scaler_data, f, indent=2)
print(f"Scaler (JSON) berhasil disimpan ke : {scaler_path}")

# 3. Simpan Metadata dan Metrik Evaluasi
metadata = {
    "ticker": TICKER,
    "target_col": TARGET_COL,
    "window_size": int(WINDOW_SIZE),
    "batch_size": int(BATCH_SIZE),
    "epochs": int(EPOCHS),
    "learning_rate": float(LEARNING_RATE),
    "train_size_ratio": float(TRAIN_SIZE_RATIO),
    "best_model": best_variant,
    "metrics": {
        "LSTM": {k: float(v) for k, v in metrics_lstm.items()},
        "GRU": {k: float(v) for k, v in metrics_gru.items()}
    }
}
metadata_path = OUTPUT_DIR / f"{TICKER}_metrics.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print(f"Metadata berhasil disimpan ke   : {metadata_path}")

# 4. Simpan SQL Seeding untuk tabel stock_models di Cloudflare D1
scaler_str = json.dumps(scaler_data).replace("'", "''")
metrics_str = json.dumps(metadata).replace("'", "''")
model_sql = (
    f"INSERT INTO stock_models (ticker, scaler_json, metrics_json) "
    f"VALUES ('{TICKER}', '{scaler_str}', '{metrics_str}') "
    f"ON CONFLICT(ticker) DO UPDATE SET scaler_json=excluded.scaler_json, metrics_json=excluded.metrics_json;\n"
)
sql_path = OUTPUT_DIR / f"{TICKER}_model_seed.sql"
with open(sql_path, "w", encoding="utf-8") as f:
    f.write(model_sql)
print(f"SQL Model Seeding berhasil disimpan ke : {sql_path}")
